# Practice 2: Transfer Learning on CIFAR-10

This notebook is the presentation and analysis layer for Practice 2. The reusable implementation remains in `processing_own_phase`; the notebook does not duplicate the training pipeline.

**Workflow:** define the problem → audit data → prevent leakage → establish a baseline → run controlled experiments → select by validation → verify the checkpoint → evaluate test once → analyze errors.

## Phase 1 - Problem Definition

| Item | Definition |
|---|---|
| Task | Supervised, single-label, 10-class image classification |
| Input | CIFAR-10 RGB image, originally `3 × 32 × 32` |
| Model input | Image resized/cropped to `3 × 224 × 224` and normalized with ImageNet statistics |
| Output | Ten raw logits, one per CIFAR-10 class |
| Primary metric | Validation/test accuracy |
| Secondary metrics | Macro F1, per-class precision and recall |
| Model family | ImageNet-pretrained CNNs from TorchVision |
| Main constraint | Training time and memory on CPU/MPS hardware |

The official CIFAR-10 test set is excluded from model and hyperparameter selection. Experiments are compared on validation data. Only the selected checkpoint is eligible for final test evaluation.

## Phase 2 - Environment Setup

The following setup works when the notebook is opened from either the project root or the `notebooks` directory.

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torchvision
from IPython.display import Image, Markdown, display

CURRENT_DIR = Path.cwd().resolve()
PROJECT_CANDIDATES = (CURRENT_DIR, CURRENT_DIR.parent)
PROJECT_ROOT = next(
    path
    for path in PROJECT_CANDIDATES
    if (path / "processing_own_phase").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs import CLASS_NAMES, CONFIG, EXPERIMENTS, OUTPUT_DIR, REPORTS_DIR, RUNS_DIR
from processing_own_phase.visualize import (
    plot_experiment_comparison,
    plot_validation_test_comparison,
)

def colored_table(dataframe, caption, gradient_columns=None, boolean_columns=None):
    """Create a colored table for notebook presentation."""
    gradient_columns = gradient_columns or []
    boolean_columns = boolean_columns or []
    styler = dataframe.style.set_caption(caption).set_table_styles([
        {"selector": "caption", "props": [
            ("caption-side", "top"), ("font-size", "17px"),
            ("font-weight", "bold"), ("color", "#5dade2"), ("padding", "10px")
        ]},
        {"selector": "thead th", "props": [
            ("background-color", "#1f4e78"), ("color", "white"),
            ("font-weight", "bold"), ("text-align", "center"),
            ("padding", "9px 13px"), ("border", "1px solid #5b9bd5")
        ]},
        {"selector": "tbody tr:nth-child(even)", "props": [
            ("background-color", "rgba(91, 155, 213, 0.15)")
        ]},
        {"selector": "tbody tr:hover", "props": [
            ("background-color", "rgba(91, 155, 213, 0.30)")
        ]},
        {"selector": "td", "props": [
            ("padding", "8px 13px"),
            ("border-bottom", "1px solid rgba(127, 127, 127, 0.35)"),
            ("text-align", "center")
        ]},
    ])
    gradient_columns = [c for c in gradient_columns if c in dataframe.columns]
    if gradient_columns:
        styler = styler.background_gradient(cmap="Blues", subset=gradient_columns)
    boolean_columns = [c for c in boolean_columns if c in dataframe.columns]
    if boolean_columns:
        styler = styler.map(
            lambda value: (
                "background-color: #198754; color: white; font-weight: bold"
                if value is True else
                "background-color: #6c757d; color: white"
            ),
            subset=boolean_columns,
        )
    return styler

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"TorchVision: {torchvision.__version__}")
print(f"Project root: {PROJECT_ROOT}")

# Persistent evidence of the accelerator used by the selected controlled run.
from processing_own_phase.utils import get_device

selection_record = json.loads(
    (OUTPUT_DIR / "controlled_experiment_selection.json").read_text()
)
selected_checkpoint = Path(selection_record["selected_checkpoint"])
training_log = next(selected_checkpoint.parent.glob("*.log"))
device_log_line = next(
    line for line in training_log.read_text().splitlines()
    if "Using device:" in line
)
training_device = device_log_line.rsplit("Using device:", 1)[1].strip().lower()

final_summary = json.loads((OUTPUT_DIR / "summary.json").read_text())
final_device = str(final_summary["device"]).lower()
runtime_device = str(get_device()).lower()
gpu_backends = {"cuda", "mps"}

assert training_device in gpu_backends, (
    f"Selected run was not GPU accelerated: {training_device}"
)
assert selection_record["test_data_used"] is False

gpu_evidence = pd.DataFrame([
    {
        "stage": "Controlled training (selected E2)",
        "evidence_source": str(training_log.relative_to(PROJECT_ROOT)),
        "device": training_device,
        "gpu_accelerated": training_device in gpu_backends,
        "meaning": "Apple GPU through Metal" if training_device == "mps" else "NVIDIA GPU through CUDA",
    },
    {
        "stage": "Final evaluation",
        "evidence_source": "outputs/summary.json",
        "device": final_device,
        "gpu_accelerated": final_device in gpu_backends,
        "meaning": "Recorded final-evaluation device",
    },
    {
        "stage": "Current notebook runtime",
        "evidence_source": "live torch backend check",
        "device": runtime_device,
        "gpu_accelerated": runtime_device in gpu_backends,
        "meaning": "Best accelerator currently available",
    },
])

display(Markdown(
    "### GPU Training Evidence\n"
    "The selected checkpoint's persistent training log records `Using device: mps`. "
    "Therefore E2 was trained on the Apple GPU through Metal Performance Shaders, "
    "not inferred only from the current runtime. This cell does not retrain the model."
))
display(colored_table(
    gpu_evidence,
    "GPU accelerator evidence",
    boolean_columns=["gpu_accelerated"],
))

## Phase 3 - Data Loading

CIFAR-10 contains 50,000 official training images and 10,000 official test images. The project creates a reproducible train/validation split from the official training pool.

- Training data uses random crop, horizontal flip, color jitter and ImageNet normalization.
- Validation and test use deterministic resize, center crop and the same normalization.
- Training and validation indices must be disjoint.
- Test data must not influence preprocessing choices, experiment selection or checkpoint selection.

CIFAR-10 is balanced, but a future revision should use an explicitly stratified split so every class count is guaranteed rather than merely expected.

In [ ]:
from processing_own_phase.data import get_class_distribution, load_datasets

train_subset, val_subset, test_dataset = load_datasets(
    data_dir=CONFIG["data_dir"],
    val_ratio=1.0 - CONFIG["train_split_ratio"],
    seed=CONFIG["seed"],
)

train_indices = set(train_subset.indices)
val_indices = set(val_subset.indices)
assert train_indices.isdisjoint(val_indices)
assert len(train_subset) == 45_000
assert len(val_subset) == 5_000
assert len(test_dataset) == 10_000

split_summary = pd.DataFrame(
    {
        "split": ["Train", "Validation", "Test"],
        "samples": [len(train_subset), len(val_subset), len(test_dataset)],
        "random_augmentation": [True, False, False],
        "used_for_selection": [False, True, False],
        "used_for_final_test": [False, False, True],
    }
)
display(Markdown(
    "**Leakage control:** split indices are determined before dataset views with "
    "preprocessing are constructed. TorchVision transforms run lazily only when "
    "an image is accessed."
))
display(colored_table(
    split_summary,
    "Dataset split summary",
    gradient_columns=["samples"],
    boolean_columns=[
        "random_augmentation",
        "used_for_selection",
        "used_for_final_test",
    ],
))

## Phase 4 - Exploratory Data Analysis

EDA is restricted to structural and visual understanding needed before modeling. CIFAR-10 contains balanced semantic classes, low-resolution RGB inputs and substantial within-class variation. The class table below confirms the split distributions; representative samples and transform inspection provide the visual sanity checks.

The official test labels are not used to make modeling decisions. Test distribution is shown only as a dataset-integrity check.

In [ ]:
class_distribution = pd.DataFrame(
    {
        "Train": get_class_distribution(train_subset),
        "Validation": get_class_distribution(val_subset),
        "Test": get_class_distribution(test_dataset),
    }
)
class_distribution.loc["Total"] = class_distribution.sum(axis=0)
display(colored_table(
    class_distribution,
    "Class distribution",
    gradient_columns=["Train", "Validation", "Test"],
))

In [ ]:
for filename, title in (
    ("class_distribution.png", "Class distribution across data splits"),
    ("data_samples.png", "Representative CIFAR-10 samples"),
    ("class_examples.png", "Examples from every CIFAR-10 class"),
):
    candidates = (OUTPUT_DIR / filename, REPORTS_DIR / filename)
    artifact_path = next((path for path in candidates if path.exists()), None)
    if artifact_path is not None:
        display(Markdown(f"### {title}"))
        display(Image(filename=str(artifact_path)))
    else:
        display(Markdown(f"_{title} artifact is not available._"))

## Phase 5 - Data Preprocessing

The pretrained backbones were optimized on ImageNet. Practice 2 therefore follows the expected ImageNet input convention instead of fitting a new scaler on CIFAR-10.

```text
CIFAR-10 image (32 × 32 RGB)
        ↓
Resize to approximately 255 × 255
        ↓
Random crop for train / center crop for validation and test
        ↓
224 × 224 tensor
        ↓
ImageNet mean/std normalization
```

The random operations belong only to the training transform. Applying them to validation or test would make metrics noisy and irreproducible.

In [ ]:
train_transform = train_subset.dataset.transform
validation_transform = val_subset.dataset.transform
test_transform = test_dataset.transform

train_transform_text = repr(train_transform)
validation_transform_text = repr(validation_transform)
test_transform_text = repr(test_transform)

assert train_subset.dataset is not val_subset.dataset
assert "RandomCrop" in train_transform_text
assert "RandomHorizontalFlip" in train_transform_text
assert "ColorJitter" in train_transform_text
assert "RandomCrop" not in validation_transform_text
assert "RandomHorizontalFlip" not in validation_transform_text
assert "ColorJitter" not in validation_transform_text
assert "CenterCrop" in validation_transform_text
assert validation_transform_text == test_transform_text

preprocessing_summary = pd.DataFrame(
    {
        "split": ["Train", "Validation", "Test"],
        "crop": ["RandomCrop", "CenterCrop", "CenterCrop"],
        "horizontal_flip": [True, False, False],
        "color_jitter": [True, False, False],
        "deterministic": [False, True, True],
        "normalization": ["ImageNet", "ImageNet", "ImageNet"],
    }
)
display(Markdown(
    "Preprocessing is attached after the split is fixed. Only Train receives "
    "random augmentation; Validation and Test share the same deterministic transform."
))
display(colored_table(
    preprocessing_summary,
    "Preprocessing by data split",
    boolean_columns=["horizontal_flip", "color_jitter", "deterministic"],
))

print("Training transform:\n", train_transform)
print("\nValidation/Test transform:\n", validation_transform)

## Phase 6 - Model Building and Baseline

A baseline is required before interpreting a more complex fine-tuning strategy.

| Baseline | Expected/observed role |
|---|---|
| Random guessing | Approximately 10% accuracy for ten balanced classes |
| Majority-class predictor | Approximately 10% because CIFAR-10 is balanced |
| ResNet18 head-only | Transfer-learning baseline: frozen feature extractor, trainable classifier only |

The head-only ResNet18 experiment is the meaningful implementation baseline for this assignment because the objective specifically concerns pretrained neural networks.

In [ ]:
from processing_own_phase.model import build_model

baseline_model = build_model("resnet18", training_mode="head_only")
total_parameters = sum(parameter.numel() for parameter in baseline_model.parameters())
trainable_parameters = sum(
    parameter.numel()
    for parameter in baseline_model.parameters()
    if parameter.requires_grad
)

baseline_summary = pd.DataFrame(
    {
        "item": ["Model", "Strategy", "Total parameters", "Trainable parameters", "Random baseline"],
        "value": ["ResNet18", "head_only", total_parameters, trainable_parameters, "10%"],
    }
)
display(colored_table(baseline_summary, "Baseline model summary"))

## Phase 7 - Model Training

Training is implemented in the reusable source modules rather than duplicated here. Every run uses `CrossEntropyLoss`, an experiment-specific optimizer, learning-rate scheduling, gradient clipping, TensorBoard logging and checkpoint management.

Three transfer-learning strategies are supported:

| Strategy | Trainable parameters |
|---|---|
| `head_only` | Classification head only |
| `partial_finetune` | Classification head plus selected final feature blocks |
| `full_finetune` | Entire pretrained network |

Validation is performed after each epoch. Under the current configuration, validation accuracy selects `best.pt`; the scheduler may still respond to validation loss.

In [ ]:
history_path = OUTPUT_DIR / "controlled_training_history.json"
selection_path = OUTPUT_DIR / "controlled_experiment_selection.json"
if not history_path.exists() or not selection_path.exists():
    raise FileNotFoundError(
        "Controlled artifacts are required. Run run_controlled_experiments() first."
    )

controlled_history = json.loads(history_path.read_text())
controlled_selection = json.loads(selection_path.read_text())
history_summary = []
for experiment, history in controlled_history.items():
    epochs_trained = len(history["val_loss"])
    assert epochs_trained > 1, f"{experiment} must contain a multi-epoch history"
    history_summary.append({
        "experiment": experiment,
        "epochs_trained": epochs_trained,
        "best_epoch": history["best_epoch"],
        "best_val_accuracy": history["best_metric"],
        "early_stopping": history["stopped_early"],
    })

display(colored_table(
    pd.DataFrame(history_summary),
    "Multi-epoch training history loaded from artifact",
    gradient_columns=["epochs_trained", "best_epoch", "best_val_accuracy"],
    boolean_columns=["early_stopping"],
))

for filename, title in (
    ("controlled_training_curves.png", "Train/validation loss and accuracy by epoch"),
    ("controlled_learning_rate.png", "Learning-rate schedule by epoch"),
):
    artifact_path = REPORTS_DIR / filename
    if not artifact_path.exists():
        raise FileNotFoundError(f"Missing learning-curve artifact: {artifact_path}")
    display(Markdown(f"### {title}"))
    display(Image(filename=str(artifact_path)))

## Phase 8 - Controlled Experiments

The official comparison contains two controlled ResNet18 experiments trained for the same multi-epoch budget. Backbone, dataset split, seed, augmentation, batch size, optimizer, learning rate, scheduler, loss and Early Stopping settings are fixed.

Therefore:

- E1 uses `head_only`.
- E2 uses `partial_finetune`.
- Fine-tuning strategy is the only controlled factor that changes.
- Test data is excluded from experiment selection.
- Repeating the controlled comparison across multiple seeds remains future work.

In [ ]:
controlled_ids = ["E1_resnet18_head", "E2_resnet18_partial"]
controlled_fields = [
    "model_name", "optimizer", "learning_rate", "batch_size", "scheduler",
    "epochs", "weight_decay", "image_size", "seed",
    "early_stopping_patience", "best_model_metric",
]
experiment_table = pd.DataFrame.from_dict(
    {experiment_id: EXPERIMENTS[experiment_id] for experiment_id in controlled_ids},
    orient="index",
)
experiment_table.index.name = "experiment"
for field in controlled_fields:
    assert experiment_table[field].nunique(dropna=False) == 1, (
        f"Controlled comparison violated: {field} differs between E1 and E2"
    )
assert experiment_table["training_mode"].nunique() == 2
display(colored_table(
    experiment_table[["training_mode", *controlled_fields]],
    "Controlled E1/E2 configuration — only fine-tuning strategy differs",
    gradient_columns=["learning_rate", "batch_size", "epochs"],
))
print("Controlled factor: training_mode = head_only versus partial_finetune")
print("All other displayed training and data settings are identical.")

## Phase 9 - Model Selection and Checkpoint Verification

Internal metric values use the `[0, 1]` convention. Multiplication by 100 happens only in the presentation layer.

The experiment table and comparison chart below are generated from the official controlled multi-epoch artifacts. The selected experiment is determined only by Validation accuracy; Test metrics are neither present nor permitted in this phase.

In [ ]:
comparison_path = OUTPUT_DIR / "controlled_experiment_comparison.csv"
selection_path = OUTPUT_DIR / "controlled_experiment_selection.json"
if not comparison_path.exists():
    raise FileNotFoundError(
        "controlled_experiment_comparison.csv is required. Run the controlled pipeline first."
    )

comparison = pd.read_csv(comparison_path)
selection = json.loads(selection_path.read_text())
required_columns = [
    "experiment", "strategy", "epochs_trained", "best_epoch",
    "best_val_accuracy", "best_val_loss", "macro_f1", "training_time",
]
assert set(required_columns).issubset(comparison.columns)
assert not any(column.lower().startswith("test") for column in comparison.columns)
assert selection["selection_metric"] == "val_accuracy"
assert selection["selection_source"] == "validation_only"
assert selection["test_data_used"] is False

comparison = comparison[required_columns].sort_values(
    "best_val_accuracy", ascending=False
).reset_index(drop=True)
display(colored_table(
    comparison,
    "Controlled experiment results — Validation only",
    gradient_columns=[
        "epochs_trained", "best_epoch", "best_val_accuracy",
        "best_val_loss", "macro_f1", "training_time",
    ],
))

selected_experiment = selection["selected_experiment"]
expected_selection = comparison.iloc[0]["experiment"]
assert selected_experiment == expected_selection
print(f"Selected experiment: {selected_experiment}")
print(f"Best validation accuracy: {selection['best_val_accuracy']:.2%}")
print("Selection source: Validation only; the Test loader was not accessed.")

controlled_plot_records = [
    {
        "exp_id": row["experiment"],
        "best_val_acc": row["best_val_accuracy"],
        "f1_score": row["macro_f1"],
        "training_time": row["training_time"],
    }
    for _, row in comparison.iterrows()
]
controlled_comparison_path = REPORTS_DIR / "controlled_experiment_comparison.png"
plot_experiment_comparison(
    controlled_plot_records,
    save_path=str(controlled_comparison_path),
)
display(Markdown("### Controlled E1/E2 visual comparison"))
display(Image(filename=str(controlled_comparison_path)))

### Checkpoint verification protocol

The original final-evaluation implementation rebuilt the selected architecture without proving that the Validation-selected checkpoint had been restored. Any metric produced by that invalid path is excluded from this notebook.

The corrected protocol is:

```text
Select E2 using validation accuracy
        ↓
Load E2 best.pt
        ↓
Re-evaluate validation and require exact accuracy agreement
        ↓
Evaluate the locked checkpoint on the official test set
```

Cell 21 reloads the checkpoint path recorded by the Validation-only selection artifact and performs Validation verification without accessing Test.

In [ ]:
from processing_own_phase.final_evaluate import verify_selected_checkpoint

verification = verify_selected_checkpoint(
    selection_path=str(OUTPUT_DIR / "controlled_experiment_selection.json"),
)
assert verification["status"] == "PASS"
assert verification["selection_metric"] == "val_accuracy"
assert verification["selection_source"] == "validation_only"
assert verification["test_accessed"] is False
checkpoint_path = Path(verification["checkpoint"])
assert checkpoint_path.name == "best.pt" and checkpoint_path.is_file()

verification_table = pd.DataFrame([
    {"Check": "Validation verification", "Value": verification["status"]},
    {"Check": "Selected experiment", "Value": verification["selected_experiment"]},
    {"Check": "Checkpoint", "Value": str(checkpoint_path)},
    {"Check": "Selection source", "Value": verification["selection_source"]},
    {"Check": "Recorded Validation Accuracy", "Value": f"{verification['recorded_val_accuracy']:.2%}"},
    {"Check": "Reloaded Validation Accuracy", "Value": f"{verification['verified_val_accuracy']:.2%}"},
    {"Check": "Validation delta", "Value": f"{verification['validation_delta']:.8f}"},
    {"Check": "Test accessed by this cell", "Value": verification["test_accessed"]},
])
display(colored_table(
    verification_table,
    "Checkpoint Verification — Validation only",
))

## Phase 10 - Final Test Evaluation

The final metrics below are read directly from `summary.json`, which is created only after the Validation-selected checkpoint passes verification. The Final Test pipeline records and asserts exactly one Test evaluation pass.

In [ ]:
from processing_own_phase.final_evaluate import validate_final_artifacts

artifact_validation = validate_final_artifacts(output_dir=str(OUTPUT_DIR))
final_summary = json.loads((OUTPUT_DIR / "summary.json").read_text())
assert final_summary["validation_verification"] == "PASS"
assert final_summary["selection_source"] == "validation_only"
assert final_summary["test_evaluation_count"] == 1
assert artifact_validation["status"] == "PASS"

final_metrics = pd.DataFrame([
    {"Metric": "Test Accuracy", "Value": f"{final_summary['test_accuracy']:.2%}"},
    {"Metric": "Test Loss", "Value": f"{final_summary['test_loss']:.6f}"},
    {"Metric": "Macro Precision", "Value": f"{final_summary['macro_precision']:.6f}"},
    {"Metric": "Macro Recall", "Value": f"{final_summary['macro_recall']:.6f}"},
    {"Metric": "Macro F1", "Value": f"{final_summary['macro_f1']:.6f}"},
    {"Metric": "Test Samples", "Value": int(final_summary["test_samples"])},
])
display(colored_table(final_metrics, "Final Test Metrics — artifact derived"))

confusion_matrix = pd.read_csv(
    OUTPUT_DIR / "confusion_matrix.csv",
    index_col=0,
).to_numpy()
confusion_samples = int(confusion_matrix.sum())
confusion_accuracy = float(confusion_matrix.trace() / confusion_samples)
assert confusion_samples == int(final_summary["test_samples"])
assert abs(confusion_accuracy - final_summary["test_accuracy"]) <= 1e-12
print(
    "Confusion-matrix consistency: PASS | "
    f"samples={confusion_samples:,} | accuracy={confusion_accuracy:.2%}"
)

off_diagonal = confusion_matrix.copy()
np.fill_diagonal(off_diagonal, 0)
largest_confusions = []
for flat_index in np.argsort(off_diagonal, axis=None)[::-1][:5]:
    true_index, predicted_index = np.unravel_index(flat_index, off_diagonal.shape)
    largest_confusions.append({
        "True class": CLASS_NAMES[true_index],
        "Predicted as": CLASS_NAMES[predicted_index],
        "Count": int(off_diagonal[true_index, predicted_index]),
        "Rate within true class": (
            off_diagonal[true_index, predicted_index]
            / confusion_matrix[true_index].sum()
        ),
    })
display(colored_table(
    pd.DataFrame(largest_confusions),
    "Largest off-diagonal confusions",
    gradient_columns=["Count", "Rate within true class"],
))

generalization_plot_path = REPORTS_DIR / "validation_test_comparison_current.png"
plot_validation_test_comparison(
    validation_accuracy=final_summary["verified_val_accuracy"],
    test_accuracy=final_summary["test_accuracy"],
    validation_loss=final_summary["validation_loss"],
    test_loss=final_summary["test_loss"],
    save_path=str(generalization_plot_path),
)
display(Markdown("### Validation–Test generalization from official artifacts"))
display(Image(filename=str(generalization_plot_path)))

classification_report = pd.read_csv(OUTPUT_DIR / "classification_report.csv")
classification_report = (
    classification_report
    .set_index("class")
    .loc[CLASS_NAMES]
    .reset_index()
)
total_samples = int(confusion_matrix.sum())
true_positive = np.diag(confusion_matrix).astype(int)
false_negative = confusion_matrix.sum(axis=1).astype(int) - true_positive
false_positive = confusion_matrix.sum(axis=0).astype(int) - true_positive
true_negative = total_samples - true_positive - false_negative - false_positive
true_positive_rate = np.divide(
    true_positive,
    true_positive + false_negative,
    out=np.zeros(len(CLASS_NAMES), dtype=float),
    where=(true_positive + false_negative) != 0,
)
false_positive_rate = np.divide(
    false_positive,
    false_positive + true_negative,
    out=np.zeros(len(CLASS_NAMES), dtype=float),
    where=(false_positive + true_negative) != 0,
)
classification_report = classification_report.assign(
    TP=true_positive,
    TN=true_negative,
    FP=false_positive,
    FN=false_negative,
    TPR=true_positive_rate,
    FPR=false_positive_rate,
)
assert np.array_equal(
    classification_report["support"].to_numpy(dtype=int),
    true_positive + false_negative,
)
assert np.allclose(classification_report["recall"], classification_report["TPR"])
assert np.all(
    classification_report[["TP", "TN", "FP", "FN"]].sum(axis=1)
    == total_samples
)
display(Markdown("### Per-class classification report"))
display(colored_table(
    classification_report,
    "Per-class classification report",
    gradient_columns=[
        "precision", "recall", "f1-score", "support",
        "TP", "TN", "FP", "FN", "TPR", "FPR",
    ],
))

predictions = pd.read_csv(OUTPUT_DIR / "predictions.csv")
probability_columns = [f"probability_{class_name}" for class_name in CLASS_NAMES]
missing_probability_columns = sorted(set(probability_columns) - set(predictions.columns))
if missing_probability_columns:
    raise RuntimeError(
        "ROC and PR curves require per-class probability columns. "
        "Regenerate the official artifacts once with the updated export pipeline. "
        f"Missing: {', '.join(missing_probability_columns)}"
    )
probabilities = predictions[probability_columns].to_numpy(dtype=float)
ground_truth = predictions["true_label_id"].to_numpy(dtype=int)
predicted_labels = predictions["predicted_label_id"].to_numpy(dtype=int)
assert probabilities.shape == (total_samples, len(CLASS_NAMES))
assert np.isfinite(probabilities).all()
assert np.all((probabilities >= 0.0) & (probabilities <= 1.0))
assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-6)
assert np.array_equal(probabilities.argmax(axis=1), predicted_labels)
assert np.array_equal(
    confusion_matrix,
    np.array([
        [np.sum((ground_truth == true_id) & (predicted_labels == predicted_id))
         for predicted_id in range(len(CLASS_NAMES))]
        for true_id in range(len(CLASS_NAMES))
    ]),
)

def binary_ranking_curves(binary_targets, scores):
    binary_targets = np.asarray(binary_targets, dtype=np.int64)
    scores = np.asarray(scores, dtype=float)
    positives = int(binary_targets.sum())
    negatives = len(binary_targets) - positives
    if positives == 0 or negatives == 0:
        raise ValueError("ROC and PR curves require positive and negative samples.")
    order = np.argsort(-scores, kind="mergesort")
    sorted_targets = binary_targets[order]
    sorted_scores = scores[order]
    threshold_indices = np.r_[np.flatnonzero(np.diff(sorted_scores)), len(scores) - 1]
    cumulative_tp = np.cumsum(sorted_targets)[threshold_indices]
    cumulative_fp = 1 + threshold_indices - cumulative_tp
    tpr = np.r_[0.0, cumulative_tp / positives]
    fpr = np.r_[0.0, cumulative_fp / negatives]
    recall = tpr
    precision = np.r_[1.0, cumulative_tp / (cumulative_tp + cumulative_fp)]
    roc_auc = np.sum(np.diff(fpr) * (tpr[:-1] + tpr[1:]) * 0.5)
    average_precision = np.sum(np.diff(recall) * precision[1:])
    return fpr, tpr, recall, precision, roc_auc, average_precision

class_curves = {}
one_hot_ground_truth = np.eye(len(CLASS_NAMES), dtype=int)[ground_truth]
for class_index, class_name in enumerate(CLASS_NAMES):
    class_curves[class_name] = binary_ranking_curves(
        one_hot_ground_truth[:, class_index],
        probabilities[:, class_index],
    )
micro_curve = binary_ranking_curves(
    one_hot_ground_truth.ravel(),
    probabilities.ravel(),
)
curve_grid = np.linspace(0.0, 1.0, 1001)
macro_tpr = np.mean([
    np.interp(curve_grid, class_curves[class_name][0], class_curves[class_name][1])
    for class_name in CLASS_NAMES
], axis=0)
macro_precision = np.mean([
    np.interp(curve_grid, class_curves[class_name][2], class_curves[class_name][3])
    for class_name in CLASS_NAMES
], axis=0)
macro_roc_auc = np.sum(
    np.diff(curve_grid) * (macro_tpr[:-1] + macro_tpr[1:]) * 0.5
)
macro_average_precision = np.mean([
    class_curves[class_name][5] for class_name in CLASS_NAMES
])

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for class_name in CLASS_NAMES:
    fpr, tpr, recall, precision, roc_auc, average_precision = class_curves[class_name]
    axes[0].plot(fpr, tpr, linewidth=1.4, label=f"{class_name} (AUC={roc_auc:.3f})")
    axes[1].plot(recall, precision, linewidth=1.4, label=f"{class_name} (AP={average_precision:.3f})")
axes[0].plot(micro_curve[0], micro_curve[1], color="black", linewidth=2.5, label=f"micro (AUC={micro_curve[4]:.3f})")
axes[0].plot(curve_grid, macro_tpr, color="navy", linestyle="--", linewidth=2.5, label=f"macro (AUC={macro_roc_auc:.3f})")
axes[0].plot([0, 1], [0, 1], color="gray", linestyle=":", linewidth=1.5)
axes[0].set(title="Multiclass ROC curves — One-vs-Rest", xlabel="False Positive Rate", ylabel="True Positive Rate", xlim=(0, 1), ylim=(0, 1.01))
axes[1].plot(micro_curve[2], micro_curve[3], color="black", linewidth=2.5, label=f"micro (AP={micro_curve[5]:.3f})")
axes[1].plot(curve_grid, macro_precision, color="navy", linestyle="--", linewidth=2.5, label=f"macro (AP={macro_average_precision:.3f})")
axes[1].set(title="Multiclass Precision–Recall curves — One-vs-Rest", xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.01))
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend(loc="lower left", fontsize=8, ncol=2)
fig.tight_layout()
display(fig)
plt.close(fig)

correct_values = predictions["is_correct"].to_numpy()
if not np.issubdtype(correct_values.dtype, np.bool_):
    correct_values = np.char.lower(correct_values.astype(str)) == "true"
correct_mask = pd.Series(correct_values, index=predictions.index, dtype=bool)
ground_truth_prediction = pd.concat([
    predictions.loc[~correct_mask].nlargest(15, "confidence"),
    predictions.loc[correct_mask].nsmallest(15, "confidence"),
])
ground_truth_prediction.index.name = "sample_index"
ground_truth_prediction = ground_truth_prediction.reset_index()
ground_truth_prediction = ground_truth_prediction.rename(columns={
    "true_label": "Ground truth",
    "predicted_label": "Prediction",
    "confidence": "Confidence",
    "is_correct": "Correct",
})[["sample_index", "Ground truth", "Prediction", "Confidence", "Correct"]]
display(Markdown("### Ground truth and prediction comparison"))
display(colored_table(
    ground_truth_prediction,
    "Ground truth vs prediction — confident errors and challenging correct cases",
    gradient_columns=["Confidence"],
    boolean_columns=["Correct"],
))

for filename, title in (
    ("metrics_bar.png", "Per-class precision, recall and F1"),
    ("confusion_matrix_raw.png", "Raw confusion matrix"),
    ("confusion_matrix_normalized.png", "Normalized confusion matrix"),
):
    display(Markdown(f"### {title}"))
    display(Image(filename=str(REPORTS_DIR / filename)))

## Phase 11 - Error Analysis and Visualization

The classification report, predictions, confusion matrices and confidence plots below were regenerated from the verified E2 checkpoint. Error analysis focuses on class-level weaknesses and confident mistakes rather than only the aggregate accuracy.

The official artifacts support the following analysis:

1. Per-class precision, recall and F1.
2. The largest off-diagonal confusion-matrix entries.
3. High-confidence incorrect predictions.
4. Low-confidence correct predictions.
5. Image-quality patterns such as blur, small objects, background shortcuts and crop sensitivity.
6. Whether common semantic pairs such as cat/dog or automobile/truck dominate the errors.

The notebook now computes the largest off-diagonal confusion pairs directly from `confusion_matrix.csv`. Each proposed improvement should address one observed root cause and change only one experimental factor before re-evaluation.

In [ ]:
for filename, title in (
    ("confidence_distribution.png", "Confidence distribution"),
    ("prediction_gallery_correct.png", "Highest-confidence correct predictions"),
    ("prediction_gallery_incorrect.png", "Highest-confidence incorrect predictions"),
):
    display(Markdown(f"### {title}"))
    display(Image(filename=str(REPORTS_DIR / filename)))

predictions = pd.read_csv(OUTPUT_DIR / "predictions.csv")
correct_mask = predictions["is_correct"]
if correct_mask.dtype != bool:
    correct_mask = correct_mask.astype(str).str.lower().eq("true")
challenging_correct = predictions[correct_mask].nsmallest(6, "confidence").copy()
challenging_correct["case"] = "Challenging correct"
confident_mistakes = predictions[~correct_mask].nlargest(6, "confidence").copy()
confident_mistakes["case"] = "Confident mistake"
prediction_grid = pd.concat([challenging_correct, confident_mistakes])

imagenet_mean = np.array([0.485, 0.456, 0.406]).reshape(1, 1, 3)
imagenet_std = np.array([0.229, 0.224, 0.225]).reshape(1, 1, 3)
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for ax, (dataset_index, row) in zip(axes.flat, prediction_grid.iterrows()):
    image_tensor, _ = test_dataset[int(dataset_index)]
    image = image_tensor.numpy().transpose(1, 2, 0)
    image = np.clip(image * imagenet_std + imagenet_mean, 0, 1)
    ax.imshow(image)
    title_color = "#198754" if row["case"] == "Challenging correct" else "#dc3545"
    ax.set_title(
        f"True: {row['true_label']}\nPred: {row['predicted_label']}\nConf: {row['confidence']:.1%}",
        color=title_color,
        fontsize=9,
    )
    ax.axis("off")
fig.suptitle(
    "Prediction Grid — challenging correct cases and confident mistakes",
    fontsize=16,
)
fig.tight_layout()
prediction_grid_path = REPORTS_DIR / "prediction_grid_mixed.png"
fig.savefig(prediction_grid_path, dpi=140, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Mixed prediction grid"))
display(Image(filename=str(prediction_grid_path)))

In [ ]:
display(Markdown(
    "**Artifact status:** PASS — summary, predictions and confusion matrix "
    "agree on sample count and accuracy."
))

### Bias–variance and training diagnostics

Learning curves should be interpreted before proposing a more complex model:

| Pattern | Interpretation | Candidate action |
|---|---|---|
| High train and validation error, small gap | Underfitting/high bias | Unfreeze more layers, train longer or improve optimization |
| Low train error, much higher validation error | Overfitting/high variance | Stronger augmentation, regularization or earlier stopping |
| Low errors and small gap | Better generalization | Confirm stability across seeds |

The official comparison now uses the same five-epoch budget for controlled E1/E2 runs. Validation peaked before the final epoch for the selected model, so retaining `best.pt` rather than the last checkpoint was appropriate. The remaining limitation is statistical stability: one seed cannot establish mean performance or variance.

### Interpretability

For CNN image classification, Grad-CAM, saliency maps or occlusion sensitivity are more appropriate than tabular-oriented tools such as VIF or manual ratio features.

A future Grad-CAM analysis should verify whether the selected model attends to the object rather than background shortcuts. This notebook does not claim interpretability evidence until real Grad-CAM artifacts are generated.

## Phase 12 - Save/Load Verification, Limitations and Conclusion

### Reproducibility and save/load checklist

- Fixed seed and saved split policy.
- Explicit train versus evaluation transforms.
- Config captured per run.
- TensorBoard events and text logs stored by run ID.
- Best checkpoint path stored in experiment metadata.
- Checkpoint contains architecture and training-strategy metadata.
- Reloaded validation accuracy checked before final test.
- Internal accuracy stored in `[0, 1]`; percentage conversion only for display.
- Future full results should include several seeds and report mean ± standard deviation.

### Conclusion

- The project implements an end-to-end transfer-learning pipeline for CIFAR-10.
- ResNet18 head-only is the implementation baseline.
- ResNet18 partial fine-tuning was selected using Validation accuracy only.
- Cell 21 reloads `best.pt` and requires exact Validation reproduction before final results are accepted.
- Cell 23 reads all Final Test metrics from official artifacts, verifies confusion-matrix consistency and visualizes the Validation–Test gap.
- Phase 11 includes both focused correct/incorrect galleries and a mixed prediction grid for error analysis.
- Invalid results from earlier unverified evaluation paths are excluded.
- Future work should measure stability across multiple seeds and extend class-level error analysis.

This notebook remains a presentation and analysis document. Training, evaluation and checkpoint logic must continue to live in the reusable source modules.